## `░ 1. Instalación de librerias`

In [ ]:
# %pip install sqlalchemy
# %pip install pandas
# %pip install oracledb 
%pip install openpyxl

#### `» 1.1 Importar librerias`

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
import oracledb

## `░ 2. Metodos globales y constantes`

#### `» 2.1 Constantes`

In [ ]:
# Conexión:

USER = 'devuser'
PASSWORD = 'D3vUs3r$2025'
HOST = '172.19.0.42'
PORT = '1521'
SID = 'INABIF02'

#### `» 2.1 Conexión a la base de datos Oracle`

In [ ]:
def get_engine():
   dsn = oracledb.makedsn(HOST, PORT, sid=SID)
   connection_string = f"oracle+oracledb://{USER}:{PASSWORD}@{dsn}"
   return create_engine(
      connection_string,
      poolclass=NullPool,  # ← AGREGADO: Sin pool para evitar problemas
      echo=False)  # Cambiar a True para debug)

#### `» 2.2 Métodos globales`

In [ ]:
# 2.2.1 Insertar DataFrame por lotes a la base de datos Oracle
def  insertar_dataframe_por_lotes(df, table_name, lote_size=1000):
   try:
         engine = get_engine()
         with engine.begin() as conn: # Maneja automáticamente commit/rollback
            df.to_sql(name=table_name, con=conn, if_exists='append', index=False, chunksize=lote_size)
            
         print(f'Datos insertados exitosamente en la tabla {table_name}.')
         return True, len(df)
   except Exception as e:
         print(f'Error al insertar datos: {e}')
         return False, 0
   finally:
         engine.dispose()
      

# 2.2.2 Ejecutar consulta y retornar DataFrame
def  execute_query(query, params=None):
   try:
         engine = get_engine()
         with engine.connect() as conn: # Cierra automáticamente la conexión
            if params:
               df = pd.read_sql_query(sql=text(query), con=conn, params=params)
            else:
               df = pd.read_sql_query(sql=text(query), con=conn)
            
         print(f'Consulta ejecutada exitosamente, filas encontradas: {len(df)}')
         return df
   except Exception as e:
         print(f"Error al ejecutar la consulta: {e}")
         return df.DataFrame()

## `░ 3. Importar datos desde un archivo CSV`

#### `» 3.1 Constantes:`

In [ ]:
FILE_PATH_MATRIZ = './data/Consolidado Resultados Evaluaciones Pre-Test.xlsx'
FILE_PATH_FICHA_27 = './data/Ficha 27 - Encuesta de satisfacción.xlsx'

#### `» 3.2 Registro de datos en la tabla SSI_ANEXOS_RESPUESTAS: Ficha 9 Diagnostico Familiar`

In [ ]:
# 3.1.1 Importar datos desde un archivo CSV
respuestas_anexo_9 = pd.read_excel(FILE_PATH_MATRIZ, sheet_name='1.2 DIAG FAM PIVOT')

In [ ]:
insertar_dataframe_por_lotes(respuestas_anexo_9, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

#### `» 3.3 Registro de datos en la tabla SSI_ANEXOS_RESPUESTAS: Ficha 10 FF-SIL`

In [ ]:
# 3.2.1 Importar datos desde un archivo CSV
respuestas_anexo_10 = pd.read_excel(FILE_PATH_MATRIZ, sheet_name='2.2 FUNC FAM PIVOT')

In [ ]:
insertar_dataframe_por_lotes(respuestas_anexo_10, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

#### `» 3.4 Registro de datos en la tabla SSI_ANEXOS_RESPUESTAS: Ficha 11 TSV`

In [ ]:
# 3.3.1 Importar datos desde un archivo CSV
respuestas_anexo_11 = pd.read_excel(FILE_PATH_MATRIZ, sheet_name='3.2 TSV_PIVOT')

In [ ]:
insertar_dataframe_por_lotes(respuestas_anexo_11, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

#### `» 3.5 Registro de datos en la tabla SSI_ANEXOS_RESPUESTAS: Ficha 27`

In [ ]:
# 3.5.1 Importar datos desde un archivo CSV
respuestas_anexo_27 = pd.read_excel(FILE_PATH_FICHA_27, sheet_name='1.1 Inabif_Act_3_Encuest_PIVOT')

In [ ]:
insertar_dataframe_por_lotes(respuestas_anexo_27, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

### `» Test:`

In [ ]:
# Test
QUERY = "SELECT * FROM SSI_ANEXOS_RESPUESTAS r WHERE r.AR_ID_RESPUESTA = :anexo_id"
df_result = execute_query(QUERY, params={'anexo_id': 1050})
df_result

In [ ]:
test_result = execute_query('''

               SELECT pv.* FROM (

                     SELECT
                        (
                           SELECT cf2.CF_CODIGO FROM (

                              SELECT 
                                 cf.CF_CODIGO,
                                 ROW_NUMBER() OVER (ORDER BY cf.CF_ID_CODIGO DESC) AS ORDEN_COD
                              FROM SSI_CODIGOS_FAMILIAS cf 
                              WHERE 
                                 cf.PF_ID_FAMILIA = f.PF_ID_FAMILIA
                                 AND cf.CF_ESTADO = 1
                                 AND cf.CF_ELIMINADO = 0

                           ) cf2
                           WHERE cf2.ORDEN_COD = 1

                        ) AS COD_FAM,

                        '-' AS NRO_VIV,

                        (
                           SELECT COUNT(1) FROM SSI_FAMILIA_INTEGRANTES i
                           WHERE 
                              i.FI_ESTADO = 1
                              AND i.FI_ELIMINADO = 0
                              AND i.FI_CUIDADOR != 1 -- * Todos excepto el cuidador
                              AND i.PF_ID_FAMILIA = f.PF_ID_FAMILIA
                        ) AS NUM_MIEM_FAM,

                        tf.CATDESCRIPCION AS TIP_FAM,
                        i.FI_TELEFONO AS NRO_TLF,	
                        td.CATDESCRIPCION AS TIP_DOC,	
                        i.FI_NUMERO_DOC AS NRO_DOC,	
                        i.FI_PRIMER_APE AS PRI_APE_FAM,	
                        i.FI_SEGUNDO_APE AS SEG_APE_FAM,	
                        i.FI_NOMBRES AS NOM_FAM,	
                        ts.CATDESCRIPCION AS SEXO,	
                        i.FI_FEC_NAC AS FEC_NAC,	
                        i.FI_EDAD AS EDAD_USU,	
                        ec.CATDESCRIPCION AS EST_CIV,
                        tp.CATDESCRIPCION AS PAR_FAM,	
                        '-' AS PAI_FAM,	

                        (
                           CASE
                              WHEN CA_TIENE_DISCAPACIDAD = 1 THEN 'SI'
                              ELSE 'NO'
                           END
                        ) AS TIE_DIS_FAM,	

                        '-' AS REG_CONADIS,
                        cd.CATDESCRIPCION AS TIP_DISC,
                        lm.CATDESCRIPCION AS LEN_MAT_FAM,	
                        '-' AS LEN_MAT_ESP_FAM,	
                        '-' AS AUT_IDE_ET_FAM,	
                        '-' AS AUT_IDE_ET_ESP_FAM,	
                        gi.CATDESCRIPCION AS GRAD_INST,	
                        co .CATDESCRIPCION AS OCU_FAM,	
                        ss .CATDESCRIPCION AS TIP_SEG_SAL,

                        -- * Fichas: 9, 10, 11
                        'Ficha-' || LPAD(p.AP_NUM_ANEXO, 2, '0') || ': ' || TRIM(p.AP_PREGUNTA) AS PREGUNTA,
                        r.AR_RESPUESTA AS RESPUESTA,
                        TO_CHAR(f.PF_FEC_REGISTRA, 'DD/MM/YYYY') AS FEC_REGISTRO_FICHA

                     FROM SSI_ZONA_INTERVENCION z
                     JOIN SSI_POTENCIALES_FAMILIAS f ON z.ZO_ID_ZONA = f.ZO_ID_ZONA
                     JOIN SSI_FAMILIA_INTEGRANTES i ON f.PF_ID_FAMILIA = i.PF_ID_FAMILIA
                     LEFT JOIN TGCATALOGO tf ON i.CA_ID_TIPO_FAMILIA = tf.IDCATALOGO
                     LEFT JOIN TGCATALOGO td ON i.CA_ID_TIPDOC = td.IDCATALOGO
                     LEFT JOIN TGCATALOGO ts ON i.CA_ID_SEXO = ts.IDCATALOGO
                     LEFT JOIN TGCATALOGO ec ON i.CA_ID_ESTADO_CIVIL = ec.IDCATALOGO
                     LEFT JOIN TGCATALOGO tp ON i.CA_ID_PARENTESCO = tp.IDCATALOGO
                     LEFT JOIN TGCATALOGO cd ON i.CA_ID_DISCAPACIDAD = cd.IDCATALOGO
                     LEFT JOIN TGCATALOGO lm ON i.CA_ID_LENGUA_MATERNA = lm.IDCATALOGO
                     LEFT JOIN TGCATALOGO gi ON i.CA_ID_GRADO_INST = gi.IDCATALOGO
                     LEFT JOIN TGCATALOGO co ON i.CA_ID_OCUPACION = co.IDCATALOGO
                     LEFT JOIN TGCATALOGO ss ON i.CA_ID_TIPO_SEGURO = ss.IDCATALOGO
                     JOIN SSI_ANEXOS_RESPUESTAS r ON f.PF_ID_FAMILIA = r.PF_ID_FAMILIA
                     JOIN SSI_ANEXOS_PREGUNTAS p ON p.AP_ID_PREGUNTA = r.AP_ID_PREGUNTA
                     JOIN SSI_ANEXO_FASES af ON p.SI_ID_SERVICIO = af.SF_ID_SERVICIO
                                             AND p.AP_NUM_ANEXO = af.SF_NUM_ANEXO
                                             AND r.SF_ID_FASE = af.SF_ID_FASE
                     WHERE 
                        f.PF_ESTADO = 1
                        AND f.PF_ELIMINADO = 0
                        AND i.FI_ESTADO = 1
                        AND i.FI_ELIMINADO = 0
                        AND i.FI_CUIDADOR = 1 -- * Cuidador principal
                        AND p.SI_ID_SERVICIO = 2 -- * PUNCHE
                        AND p.AP_NUM_ANEXO IN (9, 10, 11) -- * Diagnostico familiar, FFSIL y TSV
                        AND p.AP_NUM_GRUPO = -4 -- * Grupo resultados

                  ) f
                  PIVOT (
                     MAX(RESPUESTA)
                     FOR PREGUNTA IN ('Ficha-09: DIAGNOSTICO','Ficha-10: CALIFICACIÓN','Ficha-11: CALIFICACIÓN')
                  ) pv
                  ORDER BY pv.FEC_REGISTRO_FICHA ASC                            
                            
''')

In [ ]:
test_result.to_excel('./data/test_result_output.xlsx', index=False)